# **Day 44: Object Oriented Programming (Part 4) - Encapsulation**

**Encapsulation** means **hiding the inside details** of an object and giving only a **safe way** to use it.

**Real-life example:** An ATM machine. We can only use the buttons on the outside (withdraw, check balance). We cannot open the machine and take money directly. The machine protects its inside.

In the same way, we protect an object's data so that nobody can put a wrong value into it by mistake.

## **Public, Protected and Private in Python**
Python does not have strict "private" like some other languages. It uses **naming rules** instead:

| Name | Meaning |
|---|---|
| `name` | **Public**: anyone can use it |
| `_name` | **Protected**: "please do not touch from outside" (only a signal) |
| `__name` | **Private**: Python changes the name, so it is hard to reach from outside |

In [1]:
class Account:
    def __init__(self, holder, balance):
        self.holder = holder          # Public
        self._branch = "Midnapore"    # Protected
        self.__balance = balance      # Private

acc = Account("Soma", 5000)
print(acc.holder)
print(acc._branch)                    # Works, but we should not do this

# print(acc.__balance)                # This will raise an AttributeError
print(acc.__dict__)                   # See how Python renamed it

Soma
Midnapore
{'holder': 'Soma', '_branch': 'Midnapore', '_Account__balance': 5000}


Python renamed `__balance` to `_Account__balance`. This is called **name mangling**. It makes it harder (but not impossible) to change the value from outside.

## **Getters and Setters**
A **getter** is a method that **gives** the value. A **setter** is a method that **changes** the value, but only after **checking** it.

In [2]:
class Product:
    def __init__(self, price):
        self.__price = price

    def get_price(self):
        return self.__price

    def set_price(self, new_price):
        if new_price < 0:
            raise ValueError("Price cannot be negative")
        self.__price = new_price

p = Product(100)
p.set_price(120)
print(p.get_price())

120


## **@property: The Python Way**
In Python, we usually use `@property`. From outside, it **looks like a normal attribute**, but inside, a method runs and checks the value.

In [3]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price            # This line also goes through the setter below

    @property
    def price(self):                  # Getter
        return self._price

    @price.setter
    def price(self, value):           # Setter with a check
        if value < 0:
            raise ValueError("Price cannot be negative")
        self._price = value

    @property
    def price_with_gst(self):         # Read-only property (no setter)
        return round(self._price * 1.18, 2)

p = Product("Darjeeling tea 250g", 350)
print(p.price, p.price_with_gst)

p.price = 400                         # Looks like a normal assignment
print(p.price, p.price_with_gst)

# p.price = -5                        # This will raise a ValueError
# p.price_with_gst = 10               # This will raise an AttributeError (no setter)

350 413.0
400 472.0


## **Example from GIS: A Safe Location Class**
A GPS point must always have a **valid** latitude and longitude. With encapsulation, we make sure a wrong value can **never** be stored, whether it comes in the constructor or later.

In [4]:
class Location:
    def __init__(self, name, lat, lon):
        self.name = name
        self.lat = lat          # Goes through the setter
        self.lon = lon          # Goes through the setter

    @property
    def lat(self):
        return self._lat

    @lat.setter
    def lat(self, value):
        if not -90 <= value <= 90:
            raise ValueError(f"Latitude {value} must be between -90 and 90")
        self._lat = value

    @property
    def lon(self):
        return self._lon

    @lon.setter
    def lon(self, value):
        if not -180 <= value <= 180:
            raise ValueError(f"Longitude {value} must be between -180 and 180")
        self._lon = value

    @property
    def hemisphere(self):
        ns = "North" if self._lat >= 0 else "South"
        ew = "East" if self._lon >= 0 else "West"
        return f"{ns}-{ew}"

loc = Location("Tiger Hill", 27.0126, 88.2630)
print(loc.name, loc.lat, loc.lon, loc.hemisphere)

try:
    loc.lat = 127.0             # Wrong value
except ValueError as e:
    print("Error:", e)

print("Latitude is still:", loc.lat)

Tiger Hill 27.0126 88.263 North-East
Error: Latitude 127.0 must be between -90 and 90
Latitude is still: 27.0126


Let us understand the code above:
- `self.lat = lat` inside `__init__` does not store the value directly. It calls the **setter**, which checks the value first.
- When we tried to set 127.0, the setter raised an error and the old value stayed safe.
- `hemisphere` has only a getter, so it is **read-only**. It is calculated from the latitude and longitude every time.

## **A Bank Account with Protected Balance**

In [5]:
class BankAccount:
    MIN_BALANCE = 500

    def __init__(self, holder, opening):
        self.holder = holder
        self._balance = 0
        self._transactions = []
        self.deposit(opening)

    @property
    def balance(self):              # Only a getter: balance cannot be set directly
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
        self._transactions.append(+amount)

    def withdraw(self, amount):
        if self._balance - amount < self.MIN_BALANCE:
            raise ValueError(f"Must keep minimum balance of Rs {self.MIN_BALANCE}")
        self._balance -= amount
        self._transactions.append(-amount)

acc = BankAccount("Subrata Jana", 2000)
acc.deposit(1500)
acc.withdraw(2500)
print("Balance:", acc.balance)

try:
    acc.withdraw(1000)
except ValueError as e:
    print("Error:", e)

Balance: 1000
Error: Must keep minimum balance of Rs 500


## **Practice Problem: Temperature with Celsius and Fahrenheit**

In [6]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, f):
        self.celsius = (f - 32) * 5 / 9

t = Temperature(37)
print(t.fahrenheit)
t.fahrenheit = 50
print(round(t.celsius, 2))

98.6
10.0


In [7]:
# Quick question
# Will this line work? What will it print?
# class A:
#     def __init__(self):
#         self.__x = 1
# a = A()
# print(a._A__x)